#### The CelesTrack Dataset is very clean already, but mistaked can be happened by the dataset operators
Such as : 
- Missing values
- Duplication in the satellites
- value inconsistencies, etc

## Performing preprocssing on the satellite data
- Handle missing values
- Remove duplicates
- Structure data


In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [25]:
df = pd.read_csv('../data/01_raw/gp.csv')

In [26]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-11-24T00:38:47.626368,13.763307,0.002630,90.2213,67.0218,189.7579,232.6051,0,U,900,999,4320,0.000861,8.510000e-06,0.0
1,CALSPHERE 2,1964-063E,2025-11-23T19:18:27.313056,13.528807,0.002055,90.2361,70.9479,102.8184,16.7196,0,U,902,999,82849,0.000074,5.800000e-07,0.0
2,LCS 1,1965-034C,2025-11-23T23:38:09.501216,9.893093,0.001337,32.1445,304.2527,115.9378,244.2499,0,U,1361,999,18956,0.000832,1.500000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-11-23T16:55:04.202400,13.335800,0.007137,89.9895,212.6858,105.2414,309.9552,0,U,1512,999,93266,0.000132,7.500000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-11-23T22:01:14.414880,13.362341,0.006863,89.9085,124.4058,323.9637,161.4359,0,U,1520,999,93529,0.000215,1.200000e-06,0.0


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13410 entries, 0 to 13409
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   OBJECT_NAME          13410 non-null  object 
 1   OBJECT_ID            13410 non-null  object 
 2   EPOCH                13410 non-null  object 
 3   MEAN_MOTION          13410 non-null  float64
 4   ECCENTRICITY         13410 non-null  float64
 5   INCLINATION          13410 non-null  float64
 6   RA_OF_ASC_NODE       13410 non-null  float64
 7   ARG_OF_PERICENTER    13410 non-null  float64
 8   MEAN_ANOMALY         13410 non-null  float64
 9   EPHEMERIS_TYPE       13410 non-null  int64  
 10  CLASSIFICATION_TYPE  13410 non-null  object 
 11  NORAD_CAT_ID         13410 non-null  int64  
 12  ELEMENT_SET_NO       13410 non-null  int64  
 13  REV_AT_EPOCH         13410 non-null  int64  
 14  BSTAR                13410 non-null  float64
 15  MEAN_MOTION_DOT      13410 non-null 

----
### Check for duplicate satellite records

In [28]:
df[df.duplicated()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


- This dataset is very effective, it does not have any duplicate records.
- But, when building the final pipeline, it is still necessary to handle possible duplicates
- Here is a simple method to reliablely remove duplicates

In [29]:
df = df.drop_duplicates()

-------
### Handling missing values

In [30]:
df[df.isnull()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- By looking at the result of df.info(), there are no missing values acorss the entire dataset.
- But, df[df.isnull()] returning wierd result

  
- original dataset was "full" (had no null values),  df.isnull() mask was all False. 
- When you use this all-False mask for indexing, pandas replaces every single value with NaN, giving you a DataFrame full of NaNs.

In [31]:
# A better reliable way to check if atleast one row has misisng values or not

rows_with_null = df.isnull().any(axis=1)

In [32]:
rows_with_null

0        False
1        False
2        False
3        False
4        False
         ...  
13405    False
13406    False
13407    False
13408    False
13409    False
Length: 13410, dtype: bool

In [33]:
df[rows_with_null]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


#### As the dataset contains no missing values, still there is a need for handling them

In [34]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-11-24T00:38:47.626368,13.763307,0.002630,90.2213,67.0218,189.7579,232.6051,0,U,900,999,4320,0.000861,8.510000e-06,0.0
1,CALSPHERE 2,1964-063E,2025-11-23T19:18:27.313056,13.528807,0.002055,90.2361,70.9479,102.8184,16.7196,0,U,902,999,82849,0.000074,5.800000e-07,0.0
2,LCS 1,1965-034C,2025-11-23T23:38:09.501216,9.893093,0.001337,32.1445,304.2527,115.9378,244.2499,0,U,1361,999,18956,0.000832,1.500000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-11-23T16:55:04.202400,13.335800,0.007137,89.9895,212.6858,105.2414,309.9552,0,U,1512,999,93266,0.000132,7.500000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-11-23T22:01:14.414880,13.362341,0.006863,89.9085,124.4058,323.9637,161.4359,0,U,1520,999,93529,0.000215,1.200000e-06,0.0


#### We will select those features having 'logical' importance to be completlely NaNs free
Here are the following features and their methods to handle them

1. Object Name -> Fill missing names with 'Unknown'
2. Object ID -> Fill missing IDs with 'Unknown'
3. EPOCH ->  If Feature Deriving a new Feature like Days since launch -> Fill accorindgly, else drop the row
4. MEAN_MOTION,	ECCENTRICITY,	INCLINATION, RA_OF_ASC_NODE,	ARG_OF_PERICENTER,	MEAN_ANOMALY
-> mostly dropping the rows but can be derived from other features
5. NORAD_CAT_ID -> Fill with 'Unknown'
6. ELEMENT_SET_NO -> Fill with Mode
7. REV_AT_EPOCH,	BSTAR
-> Mostly imputation (mean / median)
8. MEAN_MOTION_DOT	-> Fill with (mean / median) or any other method (check carefully as it is imp feature)
9. MEAN_MOTION_DDOT -> Fill with mode (as most of them are 0)
10. BSTAR -> Handle carefully or just drop the rows

- Some other features that may not be used at all
1. EPHEMERIS_TYPE,	CLASSIFICATION_TYPE -> We will still use them by filling by 'mode' for missing values

-----

In [35]:
# Fill Missing object name
df['OBJECT_NAME'] = df['OBJECT_NAME'].fillna('Unknown')

In [36]:
# Fill Missing object ID 
df['OBJECT_ID']= df['OBJECT_ID'].fillna('Unknown')

In [37]:
# handle missing epoch -> For now, we will drop the missing rows, later perform feature engineering to fill it.
df = df.dropna(subset = ['EPOCH'])

In [38]:
# Fill Missing MEAN_MOTION, ECCENTRICITY, INCLINATION, RA_OF_ASC_NODE, ARG_OF_PERICENTER, MEAN_ANOMALY -> Drop the Nulls for now
# These are very important TLE paramters, filling them with an easay method is not recommended, as wrong value can lead to anomolous results
df = df.dropna(subset = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY'])

In [39]:
# Fill missing NORAD_CAT_ID -> Fill with 'Unknown'
df['NORAD_CAT_ID']= df['NORAD_CAT_ID'].fillna('Unknown')

In [40]:
# Fill missing ELEMENT_SET_NO -> Fill with imputation : Mode
df['ELEMENT_SET_NO']= df['ELEMENT_SET_NO'].fillna(df['ELEMENT_SET_NO'].mode())

In [41]:
# Fill missing REV_AT_EPOCH	 -> This is also a very important feature, missing with mean or median probably is not recommended
# So, we will drop the rows right now, later we will see more secured method

df = df.dropna(subset = ['REV_AT_EPOCH'])

In [42]:
# Fill missing BSTAR -> Very important feature, so we will drop the rows to avoid anomolous filling

df = df.dropna(subset = ['BSTAR'])

In [43]:
# Fill missing MEAN_MOTION_DOT, MEAN_MOTION_DDOT -> Same like Rev at epoch and Bstar, drop the rows
df = df.dropna(subset = ['MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT'])

In [44]:
# Fill missing EPHEMERIS_TYPE, CLASSIFICATION_TYPE -> Fill each one by imputation  : Mode
df['EPHEMERIS_TYPE'] = df['EPHEMERIS_TYPE'].fillna(df['EPHEMERIS_TYPE'].mode())

df['CLASSIFICATION_TYPE'] = df['CLASSIFICATION_TYPE'].fillna(df['CLASSIFICATION_TYPE'].mode())

---
#### Save the cleaned dataset

In [45]:
df.to_csv('../data/02_cleaned/satellites_cleaned.csv', index=False)

In [46]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-11-24T00:38:47.626368,13.763307,0.002630,90.2213,67.0218,189.7579,232.6051,0,U,900,999,4320,0.000861,8.510000e-06,0.0
1,CALSPHERE 2,1964-063E,2025-11-23T19:18:27.313056,13.528807,0.002055,90.2361,70.9479,102.8184,16.7196,0,U,902,999,82849,0.000074,5.800000e-07,0.0
2,LCS 1,1965-034C,2025-11-23T23:38:09.501216,9.893093,0.001337,32.1445,304.2527,115.9378,244.2499,0,U,1361,999,18956,0.000832,1.500000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-11-23T16:55:04.202400,13.335800,0.007137,89.9895,212.6858,105.2414,309.9552,0,U,1512,999,93266,0.000132,7.500000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-11-23T22:01:14.414880,13.362341,0.006863,89.9085,124.4058,323.9637,161.4359,0,U,1520,999,93529,0.000215,1.200000e-06,0.0


In [47]:
len(df)

13410